# Khám phá dữ liệu Chunked Parquet
Notebook này được tạo ra để view cấu trúc của file `test_chunked.parquet` sau quá trình Semantic Chunking.

In [28]:
import pandas as pd

# Đường dẫn tương đối từ thư mục notebooks
file_path = '../data/EnterpriseRAG-Bench/data/documents/test_10_markdown_chunked.parquet'
df = pd.read_parquet(file_path)

print(f"Tổng số chunks (rows): {len(df)}")
df.info()

Tổng số chunks (rows): 208
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   doc_id           208 non-null    object
 1   source_type      208 non-null    object
 2   title            208 non-null    object
 3   content          208 non-null    object
 4   chunk_id         208 non-null    int64 
 5   original_doc_id  208 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.9+ KB


### Xem thử 5 dòng đầu tiên

In [29]:
pd.set_option('display.max_colwidth', 150)
df.head()

,doc_id,source_type,title,content,chunk_id,original_doc_id
0,dsid_e54ef48bae78474684a957cf613d47d5_chunk0,confluence,Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod),"# Title: Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod) \n## Purpose\nThis runbook describes the operational procedures to deploy, upgr...",0,dsid_e54ef48bae78474684a957cf613d47d5
1,dsid_e54ef48bae78474684a957cf613d47d5_chunk1,confluence,Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod),## Quick reference\n**Service name:** perf-canary \n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- He...,1,dsid_e54ef48bae78474684a957cf613d47d5
2,dsid_e54ef48bae78474684a957cf613d47d5_chunk2,confluence,Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod),## Safety and guardrails (must-read)\n1) **Overhead budget:** perf-canary must remain below the region budget (target <0.5% of GPU capacity for co...,2,dsid_e54ef48bae78474684a957cf613d47d5
3,dsid_e54ef48bae78474684a957cf613d47d5_chunk3,confluence,Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod),"## Preconditions\nBefore deploying or upgrading perf-canary in a region, confirm:\n- You have the correct change window and announcement in #eng-r...",3,dsid_e54ef48bae78474684a957cf613d47d5
4,dsid_e54ef48bae78474684a957cf613d47d5_chunk4,confluence,Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod),## Required access:\n- Kubernetes deploy permissions for the region’s perf-canary namespace\n- Terraform apply permissions for the relevant env (p...,4,dsid_e54ef48bae78474684a957cf613d47d5


### Khảo sát một Document bị cắt
Lấy ngẫu nhiên một `original_doc_id` bị cắt làm nhiều chunk để xem thuật toán bắt Topic Shift tốt như thế nào.

In [30]:
chunk_counts = df['original_doc_id'].value_counts()
if not chunk_counts.empty and chunk_counts.iloc[0] > 1:
    sample_doc_id = chunk_counts[chunk_counts > 1].index[0]
    print(f"Khảo sát Doc ID gốc: {sample_doc_id}")
    sample_chunks = df[df['original_doc_id'] == sample_doc_id].sort_values('chunk_id')
    
    for idx, row in sample_chunks.iterrows():
        print("="*80)
        print(f"Chunk ID: {row['chunk_id']} | Độ dài (ký tự): {len(str(row['content']))}")
        print(f"Nội dung:\n{row['content']}\n")
else:
    print("Không có bài nào bị cắt (tất cả chỉ có 1 chunk). Thử giảm threshold đi!")

Khảo sát Doc ID gốc: dsid_e54ef48bae78474684a957cf613d47d5
Chunk ID: 0 | Độ dài (ký tự): 634
Nội dung:
# Title: Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)  
## Purpose
This runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.  
perf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.  
**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.  
---

Chunk ID: 1 | Độ dài (ký tự): 727
Nội dung:
## Quick reference
**Service name:** perf-canary  
**Deployment mechanism:**
- Terraform: service account / IAM / secrets / region wiring
- Helm: workload deployment and per-region config  
**Primary dashboards:**
- G